<a href="https://colab.research.google.com/github/matias-ferrero/algo2-stats/blob/main/stats_algo2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STATS

## Planilla

Cambiá esta URL para correr las estadísticas sobre otra planilla (por ejemplo, la de otro cuatrimestre).

In [ ]:
URL = 'https://docs.google.com/spreadsheets/d/1CTtpW6oQ0KYsLwz0sL75q_-51-sdZIXGb1wl09WfOng/edit?gid=0#gid=0' #@param {type:"string"}

## Configuración

### Dependencias

In [ ]:
# Imports para conectividad y autenticación con Google Sheets
import gspread as gs # API de Python para manipular Google Sheets
from google.colab import auth # Maneja la autenticación del usuario en entornos de Colab
from google.auth import default # Obtiene las credenciales por defecto para acceder a la API
from google.auth.exceptions import GoogleAuthError # Maneja errores durante el proceso de autenticación

# Imports para el logger
import logging
import sys

# Imports para manipulación y análisis de datos
import pandas as pd
import unicodedata

# Imports para visualización de datos y gráficos/figuras
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker # Formatea los ticks del eje Y como enteros prolijos

### Constantes

In [ ]:
COLUMNS = ['Padron', 'intentos', 'Hora entrega', 'Estado', 'Corrector', 'Aprobado', 'Codigo', 'Pruebas', 'Informe', 'Interaccion', 'Nota final']

In [ ]:
TP0_SHEET = "TP0"
TP1_SHEET = "TP1"
LISTA_SHEET = "LISTA"
ABB_SHEET = "ABB"
HASH_SHEET = "HASH"
TP2_SHEET = "TP2"

# La planilla de 2026+ renombró la hoja HASH a DIC; ambas se refieren al mismo TP
SHEET_ALIASES = {"DIC": HASH_SHEET}

# Solo se procesan estas hojas; cualquier otra hoja (Karma, Parametros, Final, Asignaciones, ...) se ignora
VALID_SHEETS = {TP0_SHEET, TP1_SHEET, LISTA_SHEET, ABB_SHEET, HASH_SHEET, TP2_SHEET}

### Logger

Configura el logger que usa el resto de la notebook. Los mensajes están en español, y el nivel queda en `INFO`: en una corrida sin problemas no debería imprimirse ningún log, ya que los mensajes de camino feliz (autenticación exitosa, hoja convertida a DataFrame, columnas conservadas, ...) quedan en `debug`. Solo se ven `warning`/`error`/`critical`, y únicamente ante datos u hojas inesperados (una hoja que no existe, un valor de `Aprobado`/`Estado` que no matchea ninguna categoría conocida, etc.).

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
    force=True # Asegura que la configuración se aplique en el entorno de Colab
)
logger = logging.getLogger(__name__)

### Preprocesamiento

#### Autenticación

Establece una conexión entre `Google Colab` y la API de `Google Sheets`. Este proceso requiere ***verificación de identidad del usuario*** e inicializa un cliente para gestionar las planillas.

**Pasos para autenticarse:**

1. Iniciá el login ejecutando esta sección.
2. Va a aparecer una ventana emergente; permití el proceso de login.
3. Va a aparecer una ventana de inicio de sesión de Google; seleccioná una cuenta que tenga acceso a las planillas.
4. Otorgá los permisos para autorizar a Google Cloud SDK a ver y gestionar tus archivos de Google Drive y Google Sheets.
5. Verificación: si el login se completa sin errores no vas a ver ningún mensaje en la consola (esta celda solo loggea en `debug`, silencioso por defecto) y vas a poder correr el resto del script directamente. Si algo sale mal, ahí sí vas a ver un error explicando qué pasó.

In [ ]:
try:
    logger.debug("Iniciando proceso de autenticación...")

    # Abre el prompt de OAuth2 de Google para el usuario
    auth.authenticate_user()

    # Obtiene las credenciales del entorno y autoriza al cliente de gspread
    credential, _ = default()
    client = gs.authorize(credential)

    logger.debug("Autenticación exitosa: cliente inicializado.")

except GoogleAuthError as auth_err:
    # Maneja problemas específicamente relacionados al flujo de login o a los permisos
    logger.error(f"Error de autenticación: asegurate de completar el flujo de login.\nDetalles: {auth_err}")
    raise SystemExit("Ejecución detenida: acceso no autorizado.") from None

except Exception as e:
    # Problemas de red o fallas inesperadas del sistema
    logger.critical(f"Error inesperado durante la conexión: {e}")
    raise SystemExit("Ejecución detenida por un error crítico.") from None

#### Normalización

Distintos cuatrimestres nombraron las mismas columnas de manera diferente (por ejemplo, `Nota final` vs `Nota Final`, `intentos` vs `Intentos`, `Padron` vs `Columna 1`/` gv`, `status` vs `Estado`, o `Hora entrega` dividida en `Date` (UTC) + `Hora` (UTC-3)). Esta sección mapea cada variante conocida al nombre canónico usado en `COLUMNS` —usando siempre `Hora`, que está en horario local (UTC-3) igual que la `Hora entrega` de 2025, y no `Date`, que viene en UTC— para que el resto de la notebook no necesite preocuparse por qué versión de la planilla está leyendo.

In [ ]:
def _normalize_key(value):
    """Forma en minúsculas, sin acentos y sin espacios sobrantes, usada para matchear nombres de columna."""
    ascii_value = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return ascii_value.strip().lower()

# Columnas que cambiaron de nombre entre cuatrimestres pero representan el mismo campo
COLUMN_ALIASES = {
    'gv': 'Padron',          # La hoja LISTA (2026+) usa " gv" en vez de "Padron"
    'columna 1': 'Padron',   # La hoja HASH (2025) usa "Columna 1" en vez de "Padron"
    'hora': 'Hora entrega',  # 2026+ dividió "Hora entrega" en "Date" (UTC) + "Hora" (UTC-3, la que se usaba antes)
    'status': 'Estado',      # Alias legible de la columna cruda "status" del pipeline de corrección
}

_CANONICAL_BY_KEY = {_normalize_key(col): col for col in COLUMNS}
_CANONICAL_BY_KEY.update(COLUMN_ALIASES)

def normalize_columns(df):
    """Renombra columnas para que tanto el formato de 2025 como el de 2026+ coincidan con COLUMNS."""
    rename = {
        col: _CANONICAL_BY_KEY[_normalize_key(col)]
        for col in df.columns
        if _normalize_key(col) in _CANONICAL_BY_KEY
    }
    return df.rename(columns=rename)

#### Hojas

Transforma todas las hojas disponibles en DataFrames para su posterior procesamiento y análisis.

- Ignora las hojas que no están en `VALID_SHEETS` (aplicando antes los alias de `SHEET_ALIASES`, por ejemplo `DIC` → `HASH`).
- Recorre cada hoja restante del archivo, usando la primera fila como encabezado y el resto como datos.
- Valida el contenido de cada hoja, distinguiendo entre datos activos, solo encabezados, o pestañas vacías.
- Detiene la ejecución si no se recolectaron datos (todas las hojas son inválidas).

In [ ]:
sh = client.open_by_url(URL)
csvs = sh.worksheets()

all_sheets = {}

for csv in csvs:
    title = SHEET_ALIASES.get(csv.title, csv.title)

    if title not in VALID_SHEETS:
        logger.debug(f"La hoja '{csv.title}' no es una de las hojas trackeadas. Se omite.")
        continue

    try:
        data = csv.get_all_values()

        if len(data) > 1:
            df = normalize_columns(pd.DataFrame(data[1:], columns=data[0]))
            all_sheets[title] = df

            logger.debug(f"La hoja '{csv.title}' se convirtió correctamente a DataFrame.")

        elif len(data) == 1:
            logger.warning(f"La hoja '{csv.title}' solo tiene encabezados y no tiene datos.")
        else:
            logger.debug(f"La hoja '{csv.title}' está completamente vacía.")

    except Exception as e:
        logger.error(f"No se pudo procesar la hoja '{csv.title}'.\nDetalles: {e}")

if not all_sheets:
    logger.error("No se pudo procesar ninguna hoja")
    raise SystemExit("Ejecución detenida: no se procesó ninguna hoja.")

#### Columnas

Filtra cada columna necesaria para el análisis, y solo conserva las hojas que contengan al menos una columna coincidente.

In [ ]:
sheets = {}

for title, df in all_sheets.items():
    columns = [col for col in COLUMNS if col in df.columns]

    if columns:
        sheets[title] = df[columns].copy()
        columns_str = ", ".join(columns)
        logger.debug(f"'{title}': columnas conservadas: [{columns_str}]")
    else:
        logger.warning(f"'{title}': no se encontró ninguna columna coincidente. Se omite.")

#### Vista previa

Imprime el comienzo de cada hoja después de filtrar las columnas.

In [ ]:
for title, df in sheets.items():
    print(f"Mostrando datos de la hoja: {title}")
    display(df.head(1))

## Estadísticas

### Utilidades

Esta función actúa como un formateador personalizado para las etiquetas de datos de los gráficos de torta usados en esta sección. Toma el porcentaje relativo y el tamaño total de la muestra para calcular la cantidad exacta de estudiantes por porción de cada gráfico.

También suprime las etiquetas de 0% para mejorar la visualización de los gráficos, manteniéndolos legibles.

In [ ]:
def fmt_labels(pct, total):
    if pct == 0:
        return ""

    absolute = int(round(pct/100.*total))
    return f"{pct:.1f}%\n({absolute})"

Esta función dibuja, sobre un `ax` de matplotlib ya creado, una barra por cada categoría de `counts` con la misma rampa de color secuencial introducida en Notas del TP1 (más clara = menos alumnos, más oscura = más alumnos, según el ranking "dense" de la propia cantidad de alumnos de cada barra). La reutilizan tanto Notas del TP1 como los módulos que comparan varias distribuciones lado a lado en subplots (Notas por sección del TP1, Intentos en TP1), para no repetir esa lógica en cada uno.

In [ ]:
def _draw_count_bar(ax, counts, students_count, xlabel='Nota', title=None, colors=None):
    font_style = {'family': 'serif', 'style': 'italic'}
    ink_primary = '#0b0b0b'
    gridline_color = '#e1e0d9'
    baseline_color = '#c3c2b7'

    n = len(counts)

    if colors is not None:
        # Colores fijos por categoría (por ejemplo, la paleta de estado en Estado en TP1
        # Desaprobados): acá el color identifica una categoría, no una magnitud, así que
        # no tiene sentido la rampa secuencial de abajo.
        bar_colors = colors
    else:
        # Rampa secuencial (azul, claro -> oscuro), validada con --ordinal: se resuelve a
        # 5 escalones distinguibles. Cada barra se cuantiza según el ranking "dense" de su
        # propia cantidad de alumnos (no de la categoría): las categorías con menos
        # alumnos salen más claras, las que concentran más alumnos salen más oscuras. El
        # ranking dense hace que dos categorías con la misma cantidad de alumnos
        # compartan siempre el mismo color.
        shades = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']
        dense_ranks = counts.rank(method='dense').astype(int) - 1
        n_distinct = dense_ranks.max() + 1 if n > 0 else 1
        bar_colors = [
            shades[min(len(shades) - 1, int(rank * len(shades) / n_distinct))]
            for rank in dense_ranks
        ]

    x_positions = range(n)
    bars = ax.bar(x_positions, counts.values, width=0.5, color=bar_colors, zorder=3)

    for bar, count in zip(bars, counts.values):
        pct = (count / students_count * 100) if students_count > 0 else 0
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{pct:.0f}%\n({count})",
            ha='center', va='bottom', fontsize=9.5, weight='bold', color=ink_primary,
            family=font_style['family'], linespacing=1.3, zorder=4
        )

    # Los ticks de ambos ejes van en tinta primaria (no itálica) para que el contraste
    # contra el fondo sea alto: son datos que hay que leer, no texto de marca.
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels(
        [f"{label:g}" if isinstance(label, (int, float)) else str(label) for label in counts.index],
        fontsize=11, color=ink_primary
    )

    max_count = counts.values.max() if n > 0 else 0
    # Con subtitulo (subplots comparando varias distribuciones) hace falta mas
    # aire arriba para que la etiqueta de la barra mas alta no choque con el titulo.
    headroom = 1.45 if title else 1.25
    ax.set_ylim(0, max_count * headroom + 1)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))
    for label in ax.get_yticklabels():
        label.set_fontsize(10.5)
        label.set_color(ink_primary)

    ax.grid(axis='y', color=gridline_color, linewidth=1, zorder=0)
    ax.set_axisbelow(True)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(baseline_color)
        ax.spines[spine].set_linewidth(1)

    ax.set_xlabel(xlabel, fontproperties={**font_style, 'size': 12}, color=ink_primary)
    ax.set_ylabel('Cantidad de alumnos', fontproperties={**font_style, 'size': 12}, color=ink_primary)

    if title:
        ax.set_title(title, fontproperties={**font_style, 'size': 14, 'weight': 'bold'}, color=ink_primary, pad=16)

Antes de procesar cualquier sección de estadísticas, esta celda chequea qué hojas trackeadas están realmente presentes en la planilla (por ejemplo, `ABB` no existe desde 2026). Cada sección de estadísticas consulta `hoja_disponible` y se omite sin error si su hoja no está, en vez de fallar con `KeyError`. Al implementar una sección nueva (`LISTA`, `TP2`, ...) alcanza con sumar su hoja a `HOJAS_REQUERIDAS`.

In [ ]:
HOJAS_REQUERIDAS = [TP0_SHEET, TP1_SHEET]  # sumar acá la hoja correspondiente al implementar una sección nueva (LISTA, TP2, ABB, HASH)

hoja_disponible = {hoja: hoja in sheets for hoja in HOJAS_REQUERIDAS}

for hoja, disponible in hoja_disponible.items():
    if not disponible:
        logger.warning(f"La hoja '{hoja}' no está presente en esta planilla; se omiten sus estadísticas.")

Esta función loggea un warning cuando algún valor de una columna categórica (por ejemplo `Aprobado` o `Estado`) no matcheó ninguna categoría conocida, algo que puede pasar si el vocabulario cambia de un cuatrimestre a otro (como ya pasó con `status`/`Estado` entre 2025 y 2026+). La usan los módulos que mapean esas columnas a categorías fijas.

In [ ]:
def _warn_unmapped(mapped, original, label):
    """Loggea un warning si algún valor de `original` no matcheó ninguna categoría conocida (quedó NaN en `mapped`)."""
    sin_mapear = mapped.isna()
    if sin_mapear.any():
        valores = sorted(set(original[sin_mapear].astype(str)))
        logger.warning(f"{label}: {sin_mapear.sum()} valores no reconocidos: {valores}")

Cada sección mayor de estadísticas (TP0, TP1, ...) tiene un prefijo fijo de numeración de figura, para que "Figura X.Y" sea consistente en toda la notebook (X identifica la sección, Y el orden de la figura dentro de ella). Al agregar una sección nueva alcanza con sumarle una entrada acá, siguiendo el próximo número consecutivo.

In [ ]:
SECTION_FIGURE_PREFIX = {'TP0': 1, 'TP1': 2}  # sumar la próxima sección acá (ej. 'LISTA': 3, 'TP2': 4, ...)

### TP0

#### RESULTADOS TP0

Este módulo calcula la proporción de estudiantes que aprobaron y desaprobaron el TP0, el primer trabajo práctico de la materia.

- Clasifica a cada estudiante presente en la hoja `TP0` según su columna `Aprobado`, en **Aprobado** o **Desaprobado**. Todo estudiante presente en `TP0` ya tiene una corrección cerrada, así que no hace falta una tercera categoría de pendientes.

Como resultado, el módulo genera un gráfico de torta para visualizar la proporción de cada categoría.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Reutiliza el mismo formateador de etiquetas (`fmt_labels`) y el mismo esquema de colores que los otros gráficos de esta sección, para que las tres visualizaciones sean consistentes entre sí.

In [ ]:
def tp0_results_pie_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de TP0 según su resultado. A diferencia de los módulos anteriores, no hace falta ningún cruce entre hojas: `Aprobado` se lee directamente de la propia hoja `TP0`, y esa columna no cambió de nombre entre el esquema de planilla de 2025 y el de 2026+, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP0_SHEET]:
    tp0_results_categories = ['Aprobado', 'Desaprobado']

    tp0_results = sheets[TP0_SHEET][['Padron', 'Aprobado']].copy()

    # Mapea la columna "Aprobado" a las categorías
    tp0_results['Resultado'] = tp0_results['Aprobado'].map({'Si': 'Aprobado', 'No': 'Desaprobado'})
    _warn_unmapped(tp0_results['Resultado'], tp0_results['Aprobado'], "TP0: columna 'Aprobado'")
    tp0_results.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan ambas categorías (incluso si el conteo es 0)
    tp0_results_categories_count = tp0_results['Resultado'].value_counts().reindex(tp0_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp0_results_students_count = tp0_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP0_SHEET]:
    tp0_results_pie_chart(tp0_results_categories, tp0_results_categories_count, tp0_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP0']}.1")

#### Abandonos post TP0

Este módulo identifica a los estudiantes que entregaron el TP0 pero nunca llegaron a entregar el TP1, es decir, bajas que ocurrieron entre ambos trabajos prácticos. Cruzando esos padrones con su resultado en el TP0, podemos estimar en qué proporción esas bajas se dieron después de aprobar el TP0 (posible abandono de la materia con un TP ya aprobado) o después de desaprobarlo (posible abandono tras la primera dificultad).

- Calcula la diferencia entre los padrones presentes en `TP0` y los presentes en `TP1`, para aislar a los estudiantes ausentes en TP1.
- Cruza esos padrones con su resultado (`Aprobado`) en TP0, clasificándolos en **TP0 Aprobado** o **TP0 Desaprobado**. Como estos estudiantes se seleccionan a partir de la propia hoja `TP0`, su corrección siempre está cerrada y no hace falta una tercera categoría de pendientes.
- Si todos los estudiantes que entregaron TP0 también entregaron TP1, el módulo lo informa por consola y no genera ningún gráfico.

Como resultado, el módulo devuelve un DataFrame con los datos de esos estudiantes, más un gráfico de torta para visualizar la proporción de cada resultado.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Reutiliza el mismo formateador de etiquetas (`fmt_labels`) y el mismo esquema de colores y categorías que el gráfico de la sección de Desaprobados TP1, para que ambas visualizaciones sean consistentes entre sí.

In [ ]:
def missing_tp1_pie_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Abandonos post TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección calcula la diferencia entre los padrones de TP0 y TP1 para aislar a los estudiantes ausentes en TP1, y categoriza su resultado en TP0. Tanto `Padron` como `Aprobado` son columnas comunes a ambos esquemas de planilla (2025 y 2026+): `Padron` ya queda normalizada por `normalize_columns` (a partir de `gv`/`Columna 1` según corresponda), y `Aprobado` no cambió de nombre entre cuatrimestres, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP0_SHEET] and hoja_disponible[TP1_SHEET]:
    missing_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado']

    padrones_en_tp0 = set(sheets[TP0_SHEET]['Padron'])
    padrones_en_tp1 = set(sheets[TP1_SHEET]['Padron'])
    padrones_ausentes = padrones_en_tp0 - padrones_en_tp1

    # Estudiantes que entregaron TP0 pero no TP1
    missing_tp1 = sheets[TP0_SHEET][sheets[TP0_SHEET]['Padron'].isin(padrones_ausentes)][['Padron', 'Aprobado']].copy()

    # Mapea la columna "Aprobado" a las categorías
    missing_tp1['TP0_Final'] = missing_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'})
    _warn_unmapped(missing_tp1['TP0_Final'], missing_tp1['Aprobado'], "Abandonos post TP0: columna 'Aprobado'")
    missing_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan ambas categorías (incluso si el conteo es 0)
    missing_tp1_categories_count = missing_tp1['TP0_Final'].value_counts().reindex(missing_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    missing_tp1_students_count = missing_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP0_SHEET] and hoja_disponible[TP1_SHEET]:
    if missing_tp1_students_count > 0:
        display(missing_tp1)
        print(f"\n")
        missing_tp1_pie_chart(missing_tp1_categories, missing_tp1_categories_count, missing_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP0']}.2")
    else:
        print("Todos los estudiantes que entregaron TP0 también entregaron TP1.")

### TP1

#### Resultados del TP1

Este módulo calcula la proporción de estudiantes que aprobaron, desaprobaron, o todavía no tienen nota en el TP1, replicando el mismo análisis de RESULTADOS TP0 pero para el segundo trabajo práctico.

- Clasifica a cada estudiante presente en la hoja `TP1` según su columna `Nota final`, en **Aprobado** (si tiene una nota numérica cargada), **Desaprobado** (si el valor es el string `"Desaprobado"`), o **Sin Nota** (si `Nota final` está vacío, es decir, la corrección todavía está abierta). A diferencia de TP0, acá sí pueden quedar correcciones abiertas, por eso hace falta esta tercera categoría.

Como resultado, el módulo genera un gráfico de torta para visualizar la proporción de cada categoría.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Reutiliza el mismo formateador de etiquetas (`fmt_labels`) que el resto de la notebook, y el mismo esquema de 3 colores que TP1 Aprobados y TP1 Desaprobados, ya que las tres comparten la misma estructura de tres categorías.

In [ ]:
def tp1_results_pie_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333', '#3366ff']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Resultados de TP1',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección categoriza a los estudiantes de TP1 según su resultado en `Nota final`. Al igual que en RESULTADOS TP0, no hace falta ningún cruce entre hojas: `Nota final` (2025) y `Nota Final` (2026+) coinciden vía `_normalize_key` sin necesitar un alias explícito, por lo que esta implementación funciona sin cambios contra cualquiera de los dos formatos.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_results_categories = ['Aprobado', 'Desaprobado', 'Sin Nota']

    tp1_results = sheets[TP1_SHEET][['Padron', 'Nota final']].copy()

    # Clasifica según "Nota final": el string "Desaprobado", vacío (corrección todavía abierta), o una nota numérica
    tp1_results['Resultado'] = tp1_results['Nota final'].apply(
        lambda nota: 'Desaprobado' if nota == 'Desaprobado' else ('Sin Nota' if nota == '' else 'Aprobado')
    )
    tp1_results.drop(columns=['Nota final'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp1_results_categories_count = tp1_results['Resultado'].value_counts().reindex(tp1_results_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp1_results_students_count = tp1_results_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_results_pie_chart(tp1_results_categories, tp1_results_categories_count, tp1_results_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.1")

#### Notas del TP1

Este módulo muestra la distribución de las notas numéricas obtenidas por quienes aprobaron el TP1. A diferencia de las demás secciones, acá no alcanza con una torta de dos o tres categorías: la nota final puede tomar varios valores distintos (incrementos de 0.25 entre 0 y 2 antes de 2026, o enteros entre 4 y 10 desde 2026), así que se usa un gráfico de barras con la cantidad de alumnos por cada nota.

- Toma únicamente los registros de `TP1` con una nota numérica cargada, descartando tanto los desaprobados (`Nota final == "Desaprobado"`) como las correcciones todavía abiertas (`Nota final` vacío).
- Siempre muestra **todas** las notas posibles del esquema vigente, aunque nadie haya sacado esa nota (con 0 alumnos), para que el gráfico no cambie de forma según qué notas salieron ese cuatrimestre.
- Cada barra se colorea con un degradé secuencial según su propia cantidad de alumnos (más clara = menos alumnos, más oscura = más alumnos), para que la nota más frecuente salte a la vista.
- La caja de **Referencias** reutiliza el mismo mecanismo (`plt.legend`) y los mismos parámetros de estilo que el resto de los gráficos de la notebook, con el total y el promedio en vez de categorías.

Como resultado, el módulo genera un gráfico de barras con el porcentaje y la cantidad de alumnos que sacó cada nota, más el total y el promedio en la caja de referencias.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de barras de este módulo. La rampa de color ordinal y el resto de la estética de cada barra viven en `_draw_count_bar` (ver Utilidades), reutilizado también por Notas por sección del TP1 e Intentos en TP1; esta función arma la figura, el título, y la caja de referencias con el total y el promedio.

In [ ]:
def tp1_notas_bar_chart(notas_count, students_count, mean_nota, fig_num):
  plt.figure(figsize=(10, 5.5))
  ax = plt.gca()

  _draw_count_bar(ax, notas_count, students_count, xlabel='Nota')

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas de TP1',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Caja de referencias: mismo mecanismo (plt.legend) y mismos parámetros de estilo que
  # usan RESULTADOS TP0 / Abandonos post TP0 / APROBADOS-DESAPROBADOS TP1, para que las
  # figuras de la notebook compartan una única estética de caja de referencias en vez de
  # que esta tenga la suya propia. No hay categorías que mostrar con su color, así que
  # los handles son invisibles (handlelength=0) y solo quedan las dos líneas de texto.
  promedio_txt = f"{mean_nota:.1f}" if students_count > 0 else "N/A"
  legend_labels = [f"Total: {students_count} aprobados", f"Promedio: {promedio_txt}"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.6)
  plt.show()

Esta sección filtra a los estudiantes que aprobaron el TP1, cuenta cuántos sacaron cada nota completando con 0 las notas del esquema vigente que nadie sacó, y calcula el promedio y la mediana. Como los dos esquemas de notas (0 a 2 antes de 2026, 4 a 10 desde 2026) no se solapan, alcanza con mirar la nota máxima obtenida ese cuatrimestre para saber cuál de los dos aplica.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas = sheets[TP1_SHEET][['Padron', 'Nota final']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado" y correcciones abiertas)
    tp1_notas = tp1_notas[~tp1_notas['Nota final'].isin(['', 'Desaprobado'])]

    # Google Sheets devuelve los decimales con coma (ej. "0,75") en planillas con configuración regional en español
    tp1_notas['Nota'] = pd.to_numeric(tp1_notas['Nota final'].str.replace(',', '.', regex=False))

    # Antes de 2026 las notas iban de 0 a 2 en intervalos de 0.25; desde 2026 van de 4 a 10 en intervalos de 1.
    # Como los rangos no se solapan, la nota máxima observada alcanza para inferir qué esquema aplica.
    notas_pre_2026 = [round(i * 0.25, 2) for i in range(9)]
    notas_2026_en_adelante = [float(i) for i in range(4, 11)]
    max_nota = tp1_notas['Nota'].max()
    notas_posibles = notas_2026_en_adelante if pd.isna(max_nota) or max_nota > 2 else notas_pre_2026

    # Cuenta las ocurrencias por nota, completando con 0 las notas posibles que nadie sacó
    tp1_notas_count = tp1_notas['Nota'].value_counts().reindex(notas_posibles, fill_value=0)

    # Calcula el total y el promedio de los aprobados
    tp1_notas_students_count = tp1_notas_count.sum()
    tp1_notas_mean = tp1_notas['Nota'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas_bar_chart(tp1_notas_count, tp1_notas_students_count, tp1_notas_mean, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.2")

#### Notas por sección del TP1

Este módulo desglosa la nota final de quienes aprobaron el TP1 en sus tres componentes — `Codigo`, `Pruebas` e `Informe` — para ver si el rendimiento es parejo entre secciones o si alguna concentra las notas más bajas.

- Parte de la misma población que Notas del TP1 (aprobados con nota numérica cargada) y del mismo esquema de notas ya detectado para `Nota final`, ya que las tres secciones puntúan en la misma escala (0 a 2 antes de 2026, 4 a 10 desde 2026).
- Al igual que Notas del TP1, siempre muestra todas las notas posibles del esquema vigente en cada sección, aunque nadie haya sacado esa nota, para que el gráfico no cambie de forma de un cuatrimestre a otro.

Como resultado, el módulo genera una figura con tres gráficos de barras — uno por sección, con el mismo eje Y para poder comparar alturas entre ellos — más el promedio de cada sección en la caja de referencias.

##### Implementación

Esta función arma una figura de tres subplots (uno por sección) y delega el dibujo de cada barra en `_draw_count_bar` (ver Utilidades), el mismo helper que usa Notas del TP1 — así evita repetir tres veces la lógica de la rampa de color, la grilla y los ejes. Solo agrega el título de figura, el eje Y compartido, y una caja de referencias única con el promedio de cada sección.

In [ ]:
def tp1_notas_por_seccion_bar_chart(secciones_counts, secciones_means, students_count, fig_num):
  secciones_labels = {'Codigo': 'Código', 'Pruebas': 'Pruebas', 'Informe': 'Informe'}

  fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), sharey=True)

  for ax, col in zip(axes, secciones_counts.keys()):
      _draw_count_bar(ax, secciones_counts[col], students_count, xlabel='Nota', title=secciones_labels[col])

  for ax in axes[1:]:
      ax.set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise a los
  # demas paneles; se recalcula una sola vez con el maximo real de las 3 secciones.
  max_count = max(counts.values.max() for counts in secciones_counts.values())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Notas por sección del TP1',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que el resto de la notebook (ver Notas del TP1), con un
  # promedio por sección en vez de uno solo.
  legend_labels = [f"Total: {students_count} aprobados"] + [
      f"Promedio {secciones_labels[col]}: {mean:.1f}" if students_count > 0 else f"Promedio {secciones_labels[col]}: N/A"
      for col, mean in secciones_means.items()
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.06, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.01,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.22, right=0.83, wspace=0.18)
  plt.show()

Esta sección filtra a los mismos aprobados que Notas del TP1, convierte `Codigo`, `Pruebas` e `Informe` a numérico (mismo formato de coma decimal), y cuenta cuántos alumnos sacaron cada nota en cada sección, reutilizando el `notas_posibles` ya calculado para `Nota final`.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_secciones = sheets[TP1_SHEET][['Padron', 'Nota final', 'Codigo', 'Pruebas', 'Informe']].copy()

    # Solo alumnos que aprobaron (excluye "Desaprobado" y correcciones abiertas), mismo filtro que tp1_notas
    tp1_secciones = tp1_secciones[~tp1_secciones['Nota final'].isin(['', 'Desaprobado'])]

    secciones = ['Codigo', 'Pruebas', 'Informe']
    for col in secciones:
        # Mismo formato de coma decimal que el resto de las notas
        tp1_secciones[col] = pd.to_numeric(tp1_secciones[col].str.replace(',', '.', regex=False))

    # Reutiliza el esquema de notas ya detectado para "Nota final" (mismas escalas: 0-2 antes
    # de 2026, 4-10 desde 2026), ya que las tres secciones puntúan en la misma escala
    tp1_secciones_counts = {
        col: tp1_secciones[col].value_counts().reindex(notas_posibles, fill_value=0)
        for col in secciones
    }
    tp1_secciones_means = {col: tp1_secciones[col].mean() for col in secciones}
    tp1_secciones_students_count = len(tp1_secciones)

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_notas_por_seccion_bar_chart(tp1_secciones_counts, tp1_secciones_means, tp1_secciones_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.3")

#### TP1 Aprobados

Este módulo analiza el desempeño previo (opcional) en el TP0 de quienes aprobaron el TP1. Es el complemento de TP1 Desaprobados: en vez de mirar por qué fallan, mira si venir con el TP0 aprobado (o directamente sin TP0, vía RPL) se traduce en mejores resultados en el TP1.

- Aísla a la población de estudiantes que aprobó el TP1 y, a partir de esos registros, cruza sus acciones en el TP0 para determinar si aprobaron, desaprobaron, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en el TP1 pero ausentes en los registros del TP0, y los clasifica como *"No disponible"* (N/A), para también tenerlos en cuenta.
- También asegura que el análisis siempre considere los tres resultados posibles del TP0 (**Aprobado, Desaprobado, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `TP0` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **TP0 N/A**, igual que si cada uno individualmente no tuviera registro en TP0.

Como resultado, el módulo devuelve un DataFrame con los datos importantes de los estudiantes, más un gráfico de torta para una mejor visualización de los números.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Es prácticamente idéntica a la de TP1 Desaprobados (mismas 3 categorías y colores), solo cambia el título y el número de figura.

In [ ]:
def passed_tp1_pie_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333', '#3366ff']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Aprobados en TP1 según su TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que aprobaron y categoriza esos registros, espejando la lógica de TP1 Desaprobados.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    passed_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 N/A']

    # Filtra a los estudiantes que aprobaron el TP1 (excluye "Desaprobado" y correcciones abiertas)
    passed_tp1 = sheets[TP1_SHEET][~sheets[TP1_SHEET]['Nota final'].isin(['', 'Desaprobado'])].copy()

    # Cruza con los datos de TP0 usando "Padron" y conservando "Aprobado". Si la hoja TP0 no
    # está disponible en esta planilla, se usa un DataFrame vacío: el merge no encuentra
    # coincidencias y todos los estudiantes caen en "TP0 N/A" más abajo.
    passed_tp1 = passed_tp1.merge(
        sheets.get(TP0_SHEET, pd.DataFrame(columns=['Padron', 'Aprobado']))[['Padron', 'Aprobado']],
        on='Padron',
        how='left'
    )

    # Mapea la columna "Aprobado" a las categorías
    passed_tp1['TP0_Final'] = passed_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'}).fillna('TP0 N/A')
    passed_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    passed_tp1_categories_count = passed_tp1['TP0_Final'].value_counts().reindex(passed_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    passed_tp1_students_count = passed_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    passed_tp1_pie_chart(passed_tp1_categories, passed_tp1_categories_count, passed_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.4")

#### TP1 Desaprobados

Este módulo analiza el desempeño previo (opcional) en el TP0 de quienes desaprobaron el TP1. Esto puede darnos una idea de por qué tantos estudiantes fallan en esta etapa, dado que lo único que tienen antes es el TP0 (aparte de RPL).

- Aísla a la población de estudiantes que desaprobó el TP1 y, a partir de esos registros, cruza sus acciones en el TP0 para determinar si aprobaron, desaprobaron, o nunca entregaron el trabajo anterior.
- Identifica a los estudiantes que están presentes en el TP1 pero ausentes en los registros del TP0, y los clasifica como *"No disponible"* (N/A), para también tenerlos en cuenta.
- También asegura que el análisis siempre considere los tres resultados posibles del TP0 (**Aprobado, Desaprobado, N/A**), incluso si alguno de esos grupos tiene cero estudiantes en un cuatrimestre en particular.
- Si la hoja `TP0` no está disponible en esta planilla (a diferencia del resto de las secciones, que se omiten enteras cuando falta la hoja que necesitan — ver `hoja_disponible` en Utilidades), este módulo igual corre: todos los estudiantes caen directamente en **TP0 N/A**, igual que si cada uno individualmente no tuviera registro en TP0.

Como resultado, el módulo devuelve un DataFrame con los datos importantes de los estudiantes, más un gráfico de torta para una mejor visualización de los números.

##### Implementación

Esta función gestiona la generación y visualización completa del gráfico de torta de este módulo. Encapsula todas las configuraciones estéticas y estructurales necesarias para transformar los datos procesados en la figura que se muestra a continuación.

In [ ]:
def failed_tp1_pie_chart(categories, categories_count, students_count, fig_num):
  plt.figure(figsize=(10, 5))

  colors = ['#66ff99', '#ff3333', '#3366ff']
  font_style = {'family': 'serif', 'style': 'italic'}

  patches, texts, autotexts = plt.pie(
      categories_count,
      labels=None,
      autopct=lambda p: fmt_labels(p, students_count),
      colors=colors,
      shadow={'ox': 0.03, 'shade': 0.2},
      pctdistance=0.8,
      explode=[0.05] * len(categories)
  )

  plt.setp(autotexts, size=11, weight="bold", color="black", family=font_style['family'])

  legend_labels = [
      f"{cat}: {val} ({(val/students_count)*100:.1f}%)"
      if students_count > 0 else f"{cat}: 0 (0.0%)"
      for cat, val in categories_count.items()
  ]

  plt.legend(
      patches,
      legend_labels,
      title=f"Referencias (Total: {students_count} alumnos)",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handletextpad=1.2,
      borderpad=1.5
  )

  plt.suptitle(
      'Desaprobados en TP1 según su TP0',
      x=0.5, y=0.92,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center'
  )

  plt.figtext(
      0.5, 0.10, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.04,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.85, bottom=0.22, right=0.5)

  plt.tight_layout()
  plt.show()

Esta sección filtra a los estudiantes que desaprobaron, descarta las columnas de notas (`Codigo`, `Pruebas`, `Informe`, `Nota final`) ya que no aportan nada al estar todos los registros desaprobados, y categoriza esos registros.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    failed_tp1_categories = ['TP0 Aprobado', 'TP0 Desaprobado', 'TP0 N/A']

    # Filtra a los estudiantes que desaprobaron el TP1
    failed_tp1 = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'].copy()

    # Descarta las columnas de notas: no aportan nada ya que todos los registros ya son desaprobados
    failed_tp1.drop(columns=['Codigo', 'Pruebas', 'Informe', 'Nota final'], errors='ignore', inplace=True)

    # Cruza con los datos de TP0 usando "Padron" y conservando "Aprobado". Si la hoja TP0 no
    # está disponible en esta planilla, se usa un DataFrame vacío: el merge no encuentra
    # coincidencias y todos los estudiantes caen en "TP0 N/A" más abajo.
    failed_tp1 = failed_tp1.merge(
        sheets.get(TP0_SHEET, pd.DataFrame(columns=['Padron', 'Aprobado']))[['Padron', 'Aprobado']],
        on='Padron',
        how='left'
    )

    # Mapea la columna "Aprobado" a las categorías
    failed_tp1['TP0_Final'] = failed_tp1['Aprobado'].map({'Si': 'TP0 Aprobado', 'No': 'TP0 Desaprobado'}).fillna('TP0 N/A')
    failed_tp1.drop(columns=['Aprobado'], inplace=True)

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    failed_tp1_categories_count = failed_tp1['TP0_Final'].value_counts().reindex(failed_tp1_categories, fill_value=0)

    # Calcula el total de estudiantes
    failed_tp1_students_count = failed_tp1_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    display(failed_tp1)
    print(f"\n")
    failed_tp1_pie_chart(failed_tp1_categories, failed_tp1_categories_count, failed_tp1_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.5")

#### Intentos en TP1

Este módulo compara la cantidad de `intentos` de entrega entre quienes aprobaron y quienes desaprobaron el TP1, para ver si un mayor número de reintentos se asocia con el resultado final.

- Como `intentos` puede tomar muchos valores distintos con pocas ocurrencias cada uno, se agrupa en rangos fijos (`1-3`, `4-6`, `7-10`, `11-15`, `16-20`, `21+`) en vez de graficar un valor por barra.
- Al igual que en Notas del TP1, siempre se muestran los seis rangos aunque alguno tenga 0 alumnos, para que el gráfico no cambie de forma de un cuatrimestre a otro.
- El promedio que se muestra en la caja de Referencias se calcula sobre los valores reales de `intentos` (sin binnear), no sobre los rangos.

Como resultado, el módulo genera una figura con dos gráficos de barras — Aprobados y Desaprobados, con el mismo rango de bins en el eje X — más el total y el promedio de cada grupo en la caja de referencias.

##### Implementación

También delega el dibujo de cada barra en `_draw_count_bar`, esta vez sobre una figura de dos subplots (Aprobados y Desaprobados) que comparten el eje Y y el mismo rango de bins en el eje X, para que las alturas sean directamente comparables entre los dos grupos.

In [ ]:
def tp1_intentos_bar_chart(counts_aprobados, total_aprobados, mean_aprobados, counts_desaprobados, total_desaprobados, mean_desaprobados, fig_num):
  fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

  _draw_count_bar(axes[0], counts_aprobados, total_aprobados, xlabel='Intentos', title='Aprobados')
  _draw_count_bar(axes[1], counts_desaprobados, total_desaprobados, xlabel='Intentos', title='Desaprobados')
  axes[1].set_ylabel('')

  # sharey=True hace que el ultimo set_ylim (dentro de _draw_count_bar) pise al otro
  # panel; se recalcula una sola vez con el maximo real de ambos grupos.
  max_count = max(counts_aprobados.values.max(), counts_desaprobados.values.max())
  axes[0].set_ylim(0, max_count * 1.45 + 1)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Intentos en TP1',
      x=0.5, y=0.99,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  promedio_aprobados_txt = f"{mean_aprobados:.1f}" if total_aprobados > 0 else "N/A"
  promedio_desaprobados_txt = f"{mean_desaprobados:.1f}" if total_desaprobados > 0 else "N/A"
  legend_labels = [
      f"Aprobados: {total_aprobados} (prom. {promedio_aprobados_txt})",
      f"Desaprobados: {total_desaprobados} (prom. {promedio_desaprobados_txt})",
  ]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  axes[-1].legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.74, bottom=0.24, right=0.68, wspace=0.12)
  plt.show()

Esta sección separa a `TP1` en aprobados (mismo filtro que Notas del TP1) y desaprobados (mismo filtro que TP1 Desaprobados), convierte `intentos` a numérico, y cuenta cuántos alumnos de cada grupo caen en cada rango de intentos.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    intentos_bins = [0, 3, 6, 10, 15, 20, float('inf')]
    intentos_labels = ['1-3', '4-6', '7-10', '11-15', '16-20', '21+']

    # Aprobados: mismo filtro que tp1_notas
    tp1_intentos_aprobados = sheets[TP1_SHEET][~sheets[TP1_SHEET]['Nota final'].isin(['', 'Desaprobado'])].copy()
    tp1_intentos_aprobados['intentos'] = pd.to_numeric(tp1_intentos_aprobados['intentos'].str.replace(',', '.', regex=False))
    tp1_intentos_aprobados_count = pd.cut(
        tp1_intentos_aprobados['intentos'], bins=intentos_bins, labels=intentos_labels
    ).value_counts().reindex(intentos_labels, fill_value=0)
    tp1_intentos_aprobados_total = len(tp1_intentos_aprobados)
    tp1_intentos_aprobados_mean = tp1_intentos_aprobados['intentos'].mean()

    # Desaprobados: mismo filtro que failed_tp1
    tp1_intentos_desaprobados = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'].copy()
    tp1_intentos_desaprobados['intentos'] = pd.to_numeric(tp1_intentos_desaprobados['intentos'].str.replace(',', '.', regex=False))
    tp1_intentos_desaprobados_count = pd.cut(
        tp1_intentos_desaprobados['intentos'], bins=intentos_bins, labels=intentos_labels
    ).value_counts().reindex(intentos_labels, fill_value=0)
    tp1_intentos_desaprobados_total = len(tp1_intentos_desaprobados)
    tp1_intentos_desaprobados_mean = tp1_intentos_desaprobados['intentos'].mean()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_intentos_bar_chart(
        tp1_intentos_aprobados_count, tp1_intentos_aprobados_total, tp1_intentos_aprobados_mean,
        tp1_intentos_desaprobados_count, tp1_intentos_desaprobados_total, tp1_intentos_desaprobados_mean,
        fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.6"
    )

#### Estado en TP1 Desaprobados

Este módulo mira el `Estado` que el pipeline de corrección automática le asignó a cada desaprobado del TP1, para distinguir cuántos fallaron por un problema del propio pipeline (no compila, timeout) de cuántos sí llegaron a corregirse pero no alcanzaron la nota mínima.

- Normaliza los valores de `Estado`, que cambian de vocabulario entre esquemas de planilla (`ok`/`error` en 2025, `SUCCESS`/`FAILED`/`TIMEOUT` desde 2026), a tres categorías fijas: **OK**, **Timeout** y **Error**.
- Siempre muestra las tres categorías con su propia barra aunque alguna tenga 0 alumnos (por ejemplo, `Timeout` nunca aparece en el esquema de 2025) — un gráfico de torta esconde o directamente hace desaparecer las categorías en 0, y si casi todos los registros caen en un mismo estado, la torta queda prácticamente de un solo color y se ve poco informativa.
- A diferencia de los demás gráficos de barras de la notebook, que colorean según la magnitud de cada barra, acá el color es fijo por categoría — la paleta de estado (verde/ámbar/rojo) — porque comunica severidad, no una magnitud relativa.

Como resultado, el módulo genera un gráfico de barras con la proporción de cada estado entre los desaprobados.

##### Implementación

Esta función reutiliza `_draw_count_bar` (ver Utilidades) igual que Notas del TP1, pero le pasa una lista de colores fija en vez de dejar que calcule la rampa secuencial — la paleta de estado (`good`/`warning`/`critical`, verde/ámbar/rojo), ya que acá el color identifica cada estado y no una magnitud.

In [ ]:
def tp1_estado_bar_chart(categories_count, students_count, fig_num):
  plt.figure(figsize=(9, 5.5))
  ax = plt.gca()

  # Paleta de estado (good/warning/critical) en vez de la rampa secuencial de
  # _draw_count_bar: acá el color identifica cada estado, no una magnitud.
  status_colors = {'OK': '#0ca30c', 'Timeout': '#fab219', 'Error': '#d03b3b'}
  bar_colors = [status_colors[cat] for cat in categories_count.index]

  _draw_count_bar(ax, categories_count, students_count, xlabel='Estado', colors=bar_colors)

  font_style = {'family': 'serif', 'style': 'italic'}
  ink_primary = '#0b0b0b'

  plt.suptitle(
      'Estado en TP1 Desaprobados',
      x=0.5, y=0.96,
      fontproperties={**font_style, 'size': 20, 'weight': 'bold'},
      ha='center', color=ink_primary
  )

  # Misma caja de referencias que Notas del TP1: las barras ya muestran cantidad y
  # porcentaje de cada estado, así que la caja solo necesita el total.
  legend_labels = [f"Total: {students_count} desaprobados"]
  dummy_handles = [plt.Line2D([], [], linestyle='none') for _ in legend_labels]

  ax.legend(
      dummy_handles,
      legend_labels,
      title="Referencias",
      title_fontproperties={**font_style, 'size': 13, 'weight': 'bold'},
      prop={**font_style, 'size': 12},
      loc="center left",
      bbox_to_anchor=(1.05, 0.5),
      labelspacing=1.5,
      handlelength=0,
      handletextpad=0,
      borderpad=1.5
  )

  plt.figtext(
      0.5, 0.065, f"Figura {fig_num}",
      ha="center", fontsize=11, family='serif', weight='bold', color='#333333'
  )

  plt.figtext(
      0.5, 0.015,
      "Cátedra Mendez/Pandolfo - Algoritmos y Estructuras de Datos - FIUBA",
      ha="center", fontsize=9, family='serif', style='italic', color='#555555',
      bbox={"facecolor":"orange", "alpha":0.1, "pad":5, "edgecolor":"none"}
  )

  plt.subplots_adjust(top=0.82, bottom=0.24, right=0.62)
  plt.show()

Esta sección filtra a los desaprobados del TP1 (mismo filtro que TP1 Desaprobados) y mapea su `Estado` a las tres categorías normalizadas, usando `_normalize_key` para que el mapeo no dependa de mayúsculas/minúsculas.

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_estado_categories = ['OK', 'Timeout', 'Error']

    # El pipeline de corrección devuelve vocabularios distintos según el esquema de planilla:
    # 'ok'/'error' en 2025, 'SUCCESS'/'FAILED'/'TIMEOUT' desde 2026. Se normalizan a 3 categorías fijas.
    status_aliases = {'ok': 'OK', 'success': 'OK', 'error': 'Error', 'failed': 'Error', 'timeout': 'Timeout'}

    tp1_estado = sheets[TP1_SHEET][sheets[TP1_SHEET]['Nota final'] == 'Desaprobado'][['Padron', 'Estado']].copy()
    tp1_estado_crudo = tp1_estado['Estado']
    tp1_estado['Estado'] = tp1_estado_crudo.map(lambda s: status_aliases.get(_normalize_key(s)))
    _warn_unmapped(tp1_estado['Estado'], tp1_estado_crudo, "Estado en TP1 Desaprobados: columna 'Estado'")

    # Cuenta las ocurrencias por categoría y reindexa para asegurar que existan las 3 categorías (incluso si el conteo es 0)
    tp1_estado_categories_count = tp1_estado['Estado'].value_counts().reindex(tp1_estado_categories, fill_value=0)

    # Calcula el total de estudiantes
    tp1_estado_students_count = tp1_estado_categories_count.sum()

##### Resultado

In [ ]:
if hoja_disponible[TP1_SHEET]:
    tp1_estado_bar_chart(tp1_estado_categories_count, tp1_estado_students_count, fig_num=f"{SECTION_FIGURE_PREFIX['TP1']}.7")